# 03 — Ingest Invoices into Bronze

## Purpose

Incrementally ingest the deterministic invoice history generated by the Billing System into a governed Delta Bronze table.

The notebook preserves the original invoice event contract, applies an explicit schema, captures source-file lineage, supports rescued data, generates deterministic record hashes, and validates financial reconciliation before downstream Silver transformations.

## Business Grain

One row represents one invoice event for a customer subscription and billing period.

The Bronze layer preserves invoice history exactly as received from the Billing System without applying business-level deduplication or lifecycle transformations.

## Design

- Read historical invoice JSON files from the canonical Billing System landing path.
- Apply an explicit invoice event schema.
- Validate identifiers, dates, domains, monetary calculations, balances, and lifecycle status allocations.
- Ingest incrementally with Databricks Auto Loader.
- Store the immutable events in a Delta Bronze table.
- Add operational lineage and ingestion metadata.
- Reconcile the Bronze content exactly with the source.
- Verify checkpoint-based idempotency.

## Source

- `/Volumes/workspace/revenue_leakage_bronze/landing/billing_system/invoices`

## Target

- `workspace.revenue_leakage_bronze.invoice_events`

## Expected First-Load Volume

- 26,925 historical invoice events
- 26,925 INSERT events
- 26,925 total Bronze events

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
)

SOURCE_SYSTEM = "Billing System"
SOURCE_ENTITY = "invoices"
SOURCE_FORMAT = "json"

EXPECTED_INITIAL_INVOICE_COUNT = 26_925
EXPECTED_TOTAL_INVOICE_EVENT_COUNT = 26_925

EXPECTED_OPERATION_COUNTS = {
    "INSERT": 26_925,
}

EXPECTED_INVOICE_STATUS_COUNTS = {
    "Open": 649,
    "Paid": 22_095,
    "Past Due": 3_674,
    "Voided": 507,
}

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

INVOICES_SOURCE_PATH = (
    f"{LANDING_PATH}/"
    "billing_system/invoices"
)

INVOICES_INITIAL_PATH = (
    f"{INVOICES_SOURCE_PATH}/initial_load"
)

INVOICES_SCHEMA_PATH = (
    f"{LANDING_PATH}/_schemas/"
    "bronze/billing_system/invoices"
)

INVOICES_CHECKPOINT_PATH = (
    f"{LANDING_PATH}/_checkpoints/"
    "bronze/billing_system/invoices"
)

INVOICES_BRONZE_TABLE = (
    "workspace.revenue_leakage_bronze."
    "invoice_events"
)

INVOICE_EVENT_SCHEMA = StructType([
    StructField(
        "invoice_id",
        StringType(),
        False,
    ),
    StructField(
        "subscription_id",
        StringType(),
        False,
    ),
    StructField(
        "customer_id",
        StringType(),
        False,
    ),
    StructField(
        "billing_period_start",
        DateType(),
        False,
    ),
    StructField(
        "billing_period_end",
        DateType(),
        False,
    ),
    StructField(
        "invoice_date",
        DateType(),
        False,
    ),
    StructField(
        "due_date",
        DateType(),
        False,
    ),
    StructField(
        "billing_frequency",
        StringType(),
        False,
    ),
    StructField(
        "currency",
        StringType(),
        False,
    ),
    StructField(
        "list_price_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False,
    ),
    StructField(
        "discount_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "net_subscription_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "overage_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "subtotal_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "tax_rate",
        DecimalType(5, 2),
        False,
    ),
    StructField(
        "tax_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "invoice_total_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "amount_paid",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "outstanding_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "voided_amount",
        DecimalType(12, 2),
        False,
    ),
    StructField(
        "invoice_status",
        StringType(),
        False,
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False,
    ),
    StructField(
        "operation",
        StringType(),
        False,
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False,
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False,
    ),
])

INVOICE_SOURCE_COLUMNS = (
    INVOICE_EVENT_SCHEMA.fieldNames()
)

BRONZE_METADATA_COLUMNS = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
    "_rescued_data",
]

INVOICE_BRONZE_COLUMNS = (
    INVOICE_SOURCE_COLUMNS
    + BRONZE_METADATA_COLUMNS
)

## 2. Load and Validate the Invoice Source

Load the historical invoice dataset with the explicit schema and validate expected volume, identifiers, referential integrity, billing periods, financial calculations, balance reconciliation, lifecycle status allocation, and domain values before Bronze ingestion.

In [0]:
CUSTOMERS_REFERENCE_PATH = (
    f"{LANDING_PATH}/crm/customers/initial_load"
)

SUBSCRIPTIONS_REFERENCE_PATH = (
    f"{LANDING_PATH}/"
    "subscription_system/subscriptions/initial_load"
)

CUSTOMER_REFERENCE_SCHEMA = StructType([
    StructField(
        "customer_id",
        StringType(),
        False,
    ),
])

SUBSCRIPTION_REFERENCE_SCHEMA = StructType([
    StructField(
        "subscription_id",
        StringType(),
        False,
    ),
    StructField(
        "customer_id",
        StringType(),
        False,
    ),
    StructField(
        "start_date",
        DateType(),
        False,
    ),
    StructField(
        "contracted_billing_amount",
        DecimalType(12, 2),
        False,
    ),
])

invoices_initial_source_df = (
    spark.read
    .format(SOURCE_FORMAT)
    .schema(INVOICE_EVENT_SCHEMA)
    .load(INVOICES_INITIAL_PATH)
    .withColumn(
        "_landing_batch",
        F.lit("initial_load"),
    )
)

invoices_source_df = (
    invoices_initial_source_df
)

customer_reference_df = (
    spark.read
    .format("json")
    .schema(CUSTOMER_REFERENCE_SCHEMA)
    .load(CUSTOMERS_REFERENCE_PATH)
    .select(
        F.col("customer_id").alias(
            "_reference_customer_id"
        )
    )
)

subscription_reference_df = (
    spark.read
    .format("json")
    .schema(SUBSCRIPTION_REFERENCE_SCHEMA)
    .load(SUBSCRIPTIONS_REFERENCE_PATH)
)

invoice_validation_df = (
    invoices_source_df.alias("invoice")
    .join(
        subscription_reference_df.alias(
            "subscription"
        ),
        on=(
            F.col("invoice.subscription_id")
            == F.col(
                "subscription.subscription_id"
            )
        ),
        how="left",
    )
    .select(
        "invoice.*",
        F.col(
            "subscription.customer_id"
        ).alias(
            "_subscription_customer_id"
        ),
        F.col(
            "subscription.start_date"
        ).alias(
            "_subscription_start_date"
        ),
        F.col(
            "subscription.contracted_billing_amount"
        ).alias(
            "_subscription_contracted_amount"
        ),
    )
    .join(
        customer_reference_df,
        on=(
            F.col("customer_id")
            == F.col("_reference_customer_id")
        ),
        how="left",
    )
)

required_field_is_missing = None

for column_name in INVOICE_SOURCE_COLUMNS:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == F.lit("")
        )
    )

    required_field_is_missing = (
        missing_condition
        if required_field_is_missing is None
        else required_field_is_missing
        | missing_condition
    )

invalid_identifier_condition = (
    ~F.col("invoice_id").rlike(
        r"^INV-S[0-9]{7}-[0-9]{8}$"
    )
    | ~F.col("subscription_id").rlike(
        r"^S[0-9]{7}$"
    )
    | ~F.col("customer_id").rlike(
        r"^C[0-9]{6}$"
    )
)

invalid_domain_condition = (
    ~F.col("invoice_status").isin(
        "Paid",
        "Open",
        "Past Due",
        "Voided",
    )
    | ~F.col("billing_frequency").isin(
        "Monthly",
        "Annual",
    )
    | (F.col("currency") != "USD")
    | (F.col("operation") != "INSERT")
    | ~F.col("payment_terms_days").isin(
        15,
        30,
        45,
    )
    | (F.col("discount_percentage") < 0)
    | (F.col("discount_percentage") > 100)
    | (F.col("tax_rate") < 0)
    | (F.col("tax_rate") > 25)
    | (F.col("list_price_amount") < 0)
    | (F.col("discount_amount") < 0)
    | (F.col("net_subscription_amount") < 0)
    | (F.col("overage_amount") < 0)
    | (F.col("subtotal_amount") < 0)
    | (F.col("tax_amount") < 0)
    | (F.col("invoice_total_amount") < 0)
    | (F.col("amount_paid") < 0)
    | (F.col("outstanding_amount") < 0)
    | (F.col("voided_amount") < 0)
)

invalid_date_condition = (
    (
        F.col("billing_period_start")
        > F.col("billing_period_end")
    )
    | (
        F.col("invoice_date")
        != F.col("billing_period_start")
    )
    | (
        F.col("due_date")
        < F.col("invoice_date")
    )
    | (
        F.datediff(
            F.col("due_date"),
            F.col("invoice_date"),
        )
        != F.col("payment_terms_days")
    )
    | (
        F.col("invoice_date")
        > F.col("snapshot_date")
    )
    | (
        F.to_date("event_timestamp")
        > F.col("snapshot_date")
    )
)

discount_mismatch_condition = (
    F.abs(
        F.col("discount_amount")
        - F.round(
            F.col("list_price_amount")
            * F.col("discount_percentage")
            / F.lit(100),
            2,
        )
    )
    > F.lit(0.01)
)

net_amount_mismatch_condition = (
    F.abs(
        F.col("net_subscription_amount")
        - (
            F.col("list_price_amount")
            - F.col("discount_amount")
        )
    )
    > F.lit(0.01)
)

contracted_amount_mismatch_condition = (
    F.col(
        "_subscription_contracted_amount"
    ).isNotNull()
    & (
        F.abs(
            F.col("net_subscription_amount")
            - F.col(
                "_subscription_contracted_amount"
            )
        )
        > F.lit(0.01)
    )
)

subtotal_mismatch_condition = (
    F.abs(
        F.col("subtotal_amount")
        - (
            F.col("net_subscription_amount")
            + F.col("overage_amount")
        )
    )
    > F.lit(0.01)
)

tax_mismatch_condition = (
    F.abs(
        F.col("tax_amount")
        - F.round(
            F.col("subtotal_amount")
            * F.col("tax_rate")
            / F.lit(100),
            2,
        )
    )
    > F.lit(0.01)
)

invoice_total_mismatch_condition = (
    F.abs(
        F.col("invoice_total_amount")
        - (
            F.col("subtotal_amount")
            + F.col("tax_amount")
        )
    )
    > F.lit(0.01)
)

balance_reconciliation_error_condition = (
    F.abs(
        F.col("invoice_total_amount")
        - (
            F.col("amount_paid")
            + F.col("outstanding_amount")
            + F.col("voided_amount")
        )
    )
    > F.lit(0.01)
)

invalid_status_allocation_condition = (
    (
        (F.col("invoice_status") == "Paid")
        & (
            (
                F.col("amount_paid")
                != F.col("invoice_total_amount")
            )
            | (F.col("outstanding_amount") != 0)
            | (F.col("voided_amount") != 0)
        )
    )
    | (
        F.col("invoice_status").isin(
            "Open",
            "Past Due",
        )
        & (
            (
                F.col("outstanding_amount")
                != F.col("invoice_total_amount")
            )
            | (F.col("amount_paid") != 0)
            | (F.col("voided_amount") != 0)
        )
    )
    | (
        (F.col("invoice_status") == "Voided")
        & (
            (
                F.col("voided_amount")
                != F.col("invoice_total_amount")
            )
            | (F.col("amount_paid") != 0)
            | (F.col("outstanding_amount") != 0)
        )
    )
)

invoice_source_metrics = (
    invoice_validation_df
    .agg(
        F.count("*").alias(
            "invoice_event_count"
        ),

        F.countDistinct(
            "invoice_id"
        ).alias(
            "distinct_invoice_id_count"
        ),

        F.countDistinct(
            F.struct(
                "invoice_id",
                "operation",
                "event_timestamp",
            )
        ).alias(
            "distinct_event_key_count"
        ),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_field_count"
        ),

        F.sum(
            F.when(
                invalid_identifier_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_identifier_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "_subscription_customer_id"
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias(
            "orphan_subscription_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "_reference_customer_id"
                ).isNull(),
                1,
            ).otherwise(0)
        ).alias(
            "orphan_customer_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "_subscription_customer_id"
                ).isNotNull()
                & (
                    F.col("customer_id")
                    != F.col(
                        "_subscription_customer_id"
                    )
                ),
                1,
            ).otherwise(0)
        ).alias(
            "subscription_customer_mismatch_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "_subscription_start_date"
                ).isNotNull()
                & (
                    F.col("invoice_date")
                    < F.col(
                        "_subscription_start_date"
                    )
                ),
                1,
            ).otherwise(0)
        ).alias(
            "invoice_before_subscription_count"
        ),

        F.sum(
            F.when(
                invalid_date_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_date_count"
        ),

        F.sum(
            F.when(
                invalid_domain_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_domain_count"
        ),

        F.sum(
            F.when(
                discount_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "discount_mismatch_count"
        ),

        F.sum(
            F.when(
                net_amount_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "net_amount_mismatch_count"
        ),

        F.sum(
            F.when(
                contracted_amount_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "contracted_amount_mismatch_count"
        ),

        F.sum(
            F.when(
                subtotal_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "subtotal_mismatch_count"
        ),

        F.sum(
            F.when(
                tax_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "tax_mismatch_count"
        ),

        F.sum(
            F.when(
                invoice_total_mismatch_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invoice_total_mismatch_count"
        ),

        F.sum(
            F.when(
                balance_reconciliation_error_condition,
                1,
            ).otherwise(0)
        ).alias(
            "balance_reconciliation_error_count"
        ),

        F.sum(
            F.when(
                invalid_status_allocation_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_status_allocation_count"
        ),
    )
    .first()
    .asDict()
)

invoice_event_count = int(
    invoice_source_metrics[
        "invoice_event_count"
    ]
)

distinct_invoice_id_count = int(
    invoice_source_metrics[
        "distinct_invoice_id_count"
    ]
)

distinct_event_key_count = int(
    invoice_source_metrics[
        "distinct_event_key_count"
    ]
)

duplicate_invoice_id_count = (
    invoice_event_count
    - distinct_invoice_id_count
)

duplicate_event_count = (
    invoice_event_count
    - distinct_event_key_count
)

operation_counts_df = (
    invoices_source_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

actual_operation_counts = {
    row["operation"]: int(row["count"])
    for row in operation_counts_df.collect()
}

status_counts_df = (
    invoices_source_df
    .groupBy("invoice_status")
    .count()
    .orderBy("invoice_status")
)

actual_status_counts = {
    row["invoice_status"]: int(row["count"])
    for row in status_counts_df.collect()
}

assert (
    invoice_event_count
    == EXPECTED_INITIAL_INVOICE_COUNT
), (
    "Unexpected invoice event count: "
    f"{invoice_event_count}"
)

assert duplicate_invoice_id_count == 0, (
    "Duplicate invoice IDs found: "
    f"{duplicate_invoice_id_count}"
)

assert duplicate_event_count == 0, (
    "Duplicate invoice events found: "
    f"{duplicate_event_count}"
)

assert actual_operation_counts == (
    EXPECTED_OPERATION_COUNTS
), (
    "Unexpected operation counts: "
    f"{actual_operation_counts}"
)

assert actual_status_counts == (
    EXPECTED_INVOICE_STATUS_COUNTS
), (
    "Unexpected invoice status counts: "
    f"{actual_status_counts}"
)

validation_error_columns = [
    "null_required_field_count",
    "invalid_identifier_count",
    "orphan_subscription_count",
    "orphan_customer_count",
    "subscription_customer_mismatch_count",
    "invoice_before_subscription_count",
    "invalid_date_count",
    "invalid_domain_count",
    "discount_mismatch_count",
    "net_amount_mismatch_count",
    "contracted_amount_mismatch_count",
    "subtotal_mismatch_count",
    "tax_mismatch_count",
    "invoice_total_mismatch_count",
    "balance_reconciliation_error_count",
    "invalid_status_allocation_count",
]

for error_column in validation_error_columns:
    error_count = int(
        invoice_source_metrics[error_column]
    )

    assert error_count == 0, (
        f"{error_column}: {error_count}"
    )

print(
    f"Invoices loaded: "
    f"{invoice_event_count:,}"
)

print(
    f"Distinct invoice IDs: "
    f"{distinct_invoice_id_count:,}"
)

print(
    f"Distinct source event keys: "
    f"{distinct_event_key_count:,}"
)

print(
    f"Duplicate invoice IDs: "
    f"{duplicate_invoice_id_count:,}"
)

print(
    f"Duplicate source events: "
    f"{duplicate_event_count:,}"
)

print(
    "Null required fields: "
    f"{invoice_source_metrics['null_required_field_count']:,}"
)

print(
    "Invalid identifiers: "
    f"{invoice_source_metrics['invalid_identifier_count']:,}"
)

print(
    "Orphan subscriptions: "
    f"{invoice_source_metrics['orphan_subscription_count']:,}"
)

print(
    "Orphan customers: "
    f"{invoice_source_metrics['orphan_customer_count']:,}"
)

print(
    "Subscription/customer mismatches: "
    f"{invoice_source_metrics['subscription_customer_mismatch_count']:,}"
)

print(
    "Invoices before subscription start: "
    f"{invoice_source_metrics['invoice_before_subscription_count']:,}"
)

print(
    "Invalid dates or billing periods: "
    f"{invoice_source_metrics['invalid_date_count']:,}"
)

print(
    "Invalid domain values: "
    f"{invoice_source_metrics['invalid_domain_count']:,}"
)

print(
    "Discount mismatches: "
    f"{invoice_source_metrics['discount_mismatch_count']:,}"
)

print(
    "Net amount mismatches: "
    f"{invoice_source_metrics['net_amount_mismatch_count']:,}"
)

print(
    "Contracted amount mismatches: "
    f"{invoice_source_metrics['contracted_amount_mismatch_count']:,}"
)

print(
    "Subtotal mismatches: "
    f"{invoice_source_metrics['subtotal_mismatch_count']:,}"
)

print(
    "Tax mismatches: "
    f"{invoice_source_metrics['tax_mismatch_count']:,}"
)

print(
    "Invoice total mismatches: "
    f"{invoice_source_metrics['invoice_total_mismatch_count']:,}"
)

print(
    "Balance reconciliation errors: "
    f"{invoice_source_metrics['balance_reconciliation_error_count']:,}"
)

print(
    "Invalid status allocations: "
    f"{invoice_source_metrics['invalid_status_allocation_count']:,}"
)

display(
    invoices_source_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias(
            "invoice_total_amount"
        ),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias(
            "amount_paid"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias(
            "voided_amount"
        ),
    )
    .orderBy("invoice_status")
)

## 3. Ingest Invoice Events into Bronze

Use Databricks Auto Loader to incrementally ingest the historical invoice JSON files into a Delta Bronze table. Preserve the complete source contract and add file lineage, ingestion metadata, rescued-data support, and a deterministic record hash.

In [0]:
invoice_record_hash_columns = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>"),
    )
    for column_name in INVOICE_SOURCE_COLUMNS
]

invoice_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        SOURCE_FORMAT,
    )
    .option(
        "cloudFiles.schemaLocation",
        INVOICES_SCHEMA_PATH,
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue",
    )
    .option(
        "rescuedDataColumn",
        "_rescued_data",
    )
    .schema(INVOICE_EVENT_SCHEMA)
    .load(INVOICES_SOURCE_PATH)
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM),
    )
    .withColumn(
        "_source_entity",
        F.lit(SOURCE_ENTITY),
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path"),
    )
    .withColumn(
        "_source_file_name",
        F.col("_metadata.file_name"),
    )
    .withColumn(
        "_source_file_size",
        F.col("_metadata.file_size"),
    )
    .withColumn(
        "_source_file_modification_time",
        F.col("_metadata.file_modification_time"),
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp(),
    )
    .withColumn(
        "_ingestion_date",
        F.to_date("_ingested_at"),
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *invoice_record_hash_columns,
            ),
            256,
        ),
    )
    .select(
        *INVOICE_BRONZE_COLUMNS
    )
)

invoice_bronze_query = (
    invoice_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        INVOICES_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        INVOICES_BRONZE_TABLE
    )
)

invoice_bronze_query.awaitTermination()

print(
    "Invoice Auto Loader ingestion completed."
)

print(
    f"Bronze table: "
    f"{INVOICES_BRONZE_TABLE}"
)

print(
    f"Canonical source: "
    f"{INVOICES_SOURCE_PATH}"
)

print(
    f"Checkpoint: "
    f"{INVOICES_CHECKPOINT_PATH}"
)

## 4. Validate and Reconcile the Bronze Invoice Table

Validate Bronze invoice volume, event uniqueness, required source and metadata fields, rescued data, canonical lineage, operation and status distributions, Delta format, and exact source-to-Bronze content reconciliation.

In [0]:
bronze_invoices_df = spark.table(
    INVOICES_BRONZE_TABLE
)

required_metadata_columns = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
]

required_metadata_is_missing = None

for column_name in required_metadata_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == F.lit("")
        )
    )

    required_metadata_is_missing = (
        missing_condition
        if required_metadata_is_missing is None
        else required_metadata_is_missing
        | missing_condition
    )

invalid_lineage_condition = (
    (F.col("_source_system") != SOURCE_SYSTEM)
    | (F.col("_source_entity") != SOURCE_ENTITY)
    | (
        ~F.col("_source_file_path").contains(
            INVOICES_SOURCE_PATH
        )
    )
)

bronze_invoice_metrics = (
    bronze_invoices_df
    .agg(
        F.count("*").alias(
            "bronze_invoice_event_count"
        ),

        F.countDistinct(
            "invoice_id"
        ).alias(
            "distinct_invoice_id_count"
        ),

        F.countDistinct(
            F.struct(
                "invoice_id",
                "operation",
                "event_timestamp",
            )
        ).alias(
            "distinct_event_key_count"
        ),

        F.countDistinct(
            "_record_hash"
        ).alias(
            "distinct_record_hash_count"
        ),

        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_source_field_count"
        ),

        F.sum(
            F.when(
                required_metadata_is_missing,
                1,
            ).otherwise(0)
        ).alias(
            "null_required_metadata_field_count"
        ),

        F.sum(
            F.when(
                F.col("_rescued_data").isNotNull()
                & (
                    F.trim(F.col("_rescued_data"))
                    != F.lit("")
                ),
                1,
            ).otherwise(0)
        ).alias(
            "rescued_data_row_count"
        ),

        F.sum(
            F.when(
                invalid_lineage_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_lineage_row_count"
        ),

        F.sum(
            F.when(
                F.col("_source_file_path").contains(
                    "/initial_load/"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "initial_load_event_count"
        ),
    )
    .first()
    .asDict()
)

source_reconciliation_df = (
    invoices_source_df
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<NULL>"),
                    )
                    for column_name
                    in INVOICE_SOURCE_COLUMNS
                ],
            ),
            256,
        ),
    )
    .select(
        "invoice_id",
        "operation",
        "event_timestamp",
        "_record_hash",
    )
)

bronze_reconciliation_df = (
    bronze_invoices_df
    .select(
        "invoice_id",
        "operation",
        "event_timestamp",
        "_record_hash",
    )
)

source_missing_from_bronze_count = (
    source_reconciliation_df
    .exceptAll(bronze_reconciliation_df)
    .count()
)

unexpected_bronze_record_count = (
    bronze_reconciliation_df
    .exceptAll(source_reconciliation_df)
    .count()
)

source_bronze_mismatch_count = (
    source_missing_from_bronze_count
    + unexpected_bronze_record_count
)

bronze_operation_counts_df = (
    bronze_invoices_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

actual_bronze_operation_counts = {
    row["operation"]: int(row["count"])
    for row in bronze_operation_counts_df.collect()
}

bronze_status_counts_df = (
    bronze_invoices_df
    .groupBy("invoice_status")
    .count()
    .orderBy("invoice_status")
)

actual_bronze_status_counts = {
    row["invoice_status"]: int(row["count"])
    for row in bronze_status_counts_df.collect()
}

target_detail = (
    spark.sql(
        f"DESCRIBE DETAIL {INVOICES_BRONZE_TABLE}"
    )
    .select("format")
    .first()
)

target_format = target_detail["format"].lower()

bronze_invoice_event_count = int(
    bronze_invoice_metrics[
        "bronze_invoice_event_count"
    ]
)

distinct_invoice_id_count = int(
    bronze_invoice_metrics[
        "distinct_invoice_id_count"
    ]
)

distinct_event_key_count = int(
    bronze_invoice_metrics[
        "distinct_event_key_count"
    ]
)

distinct_record_hash_count = int(
    bronze_invoice_metrics[
        "distinct_record_hash_count"
    ]
)

assert (
    bronze_invoice_event_count
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), (
    "Unexpected Bronze invoice count: "
    f"{bronze_invoice_event_count}"
)

assert (
    distinct_invoice_id_count
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), "Duplicate Bronze invoice IDs found."

assert (
    distinct_event_key_count
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), "Duplicate Bronze invoice event keys found."

assert (
    distinct_record_hash_count
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), "Duplicate Bronze invoice record hashes found."

assert (
    int(
        bronze_invoice_metrics[
            "null_required_source_field_count"
        ]
    )
    == 0
), "Null required source fields found."

assert (
    int(
        bronze_invoice_metrics[
            "null_required_metadata_field_count"
        ]
    )
    == 0
), "Null required metadata fields found."

assert (
    int(
        bronze_invoice_metrics[
            "rescued_data_row_count"
        ]
    )
    == 0
), "Rescued-data rows found."

assert (
    int(
        bronze_invoice_metrics[
            "invalid_lineage_row_count"
        ]
    )
    == 0
), "Invalid canonical lineage rows found."

assert (
    int(
        bronze_invoice_metrics[
            "initial_load_event_count"
        ]
    )
    == EXPECTED_INITIAL_INVOICE_COUNT
), "Unexpected initial-load count."

assert (
    actual_bronze_operation_counts
    == EXPECTED_OPERATION_COUNTS
), (
    "Unexpected Bronze operation counts: "
    f"{actual_bronze_operation_counts}"
)

assert (
    actual_bronze_status_counts
    == EXPECTED_INVOICE_STATUS_COUNTS
), (
    "Unexpected Bronze status counts: "
    f"{actual_bronze_status_counts}"
)

assert source_bronze_mismatch_count == 0, (
    "Source/Bronze content mismatches: "
    f"{source_bronze_mismatch_count}"
)

assert target_format == "delta", (
    f"Unexpected target format: {target_format}"
)

print(
    f"Bronze invoice events: "
    f"{bronze_invoice_event_count:,}"
)

print(
    f"Distinct Bronze invoice IDs: "
    f"{distinct_invoice_id_count:,}"
)

print(
    f"Distinct Bronze event keys: "
    f"{distinct_event_key_count:,}"
)

print(
    f"Distinct record hashes: "
    f"{distinct_record_hash_count:,}"
)

print(
    "Null required source fields: "
    f"{bronze_invoice_metrics['null_required_source_field_count']:,}"
)

print(
    "Null required metadata fields: "
    f"{bronze_invoice_metrics['null_required_metadata_field_count']:,}"
)

print(
    "Rescued-data rows: "
    f"{bronze_invoice_metrics['rescued_data_row_count']:,}"
)

print(
    "Invalid canonical lineage rows: "
    f"{bronze_invoice_metrics['invalid_lineage_row_count']:,}"
)

print(
    "Initial-load events: "
    f"{bronze_invoice_metrics['initial_load_event_count']:,}"
)

print(
    f"Source/Bronze content mismatches: "
    f"{source_bronze_mismatch_count:,}"
)

print(
    f"Target format: {target_format}"
)

display(
    bronze_invoices_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias(
            "invoice_total_amount"
        ),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias(
            "amount_paid"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias(
            "voided_amount"
        ),
    )
    .orderBy("invoice_status")
)

## 5. Validate Auto Loader Idempotency

Rerun the invoice ingestion using the existing Auto Loader checkpoint and confirm that previously processed invoice files are not ingested again.

In [0]:
rows_before_idempotency_rerun = (
    spark.table(INVOICES_BRONZE_TABLE)
    .count()
)

invoice_idempotency_query = (
    invoice_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        INVOICES_CHECKPOINT_PATH,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        INVOICES_BRONZE_TABLE
    )
)

invoice_idempotency_query.awaitTermination()

rows_after_idempotency_rerun = (
    spark.table(INVOICES_BRONZE_TABLE)
    .count()
)

rows_added_during_rerun = (
    rows_after_idempotency_rerun
    - rows_before_idempotency_rerun
)

assert (
    rows_before_idempotency_rerun
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), (
    "Unexpected row count before rerun: "
    f"{rows_before_idempotency_rerun}"
)

assert (
    rows_after_idempotency_rerun
    == EXPECTED_TOTAL_INVOICE_EVENT_COUNT
), (
    "Unexpected row count after rerun: "
    f"{rows_after_idempotency_rerun}"
)

assert rows_added_during_rerun == 0, (
    "Auto Loader idempotency failed. "
    f"Rows added during rerun: "
    f"{rows_added_during_rerun}"
)

print(
    "Rows before idempotency rerun: "
    f"{rows_before_idempotency_rerun:,}"
)

print(
    "Rows after idempotency rerun: "
    f"{rows_after_idempotency_rerun:,}"
)

print(
    "Rows added during rerun: "
    f"{rows_added_during_rerun:,}"
)

print(
    "Invoice Bronze ingestion is idempotent."
)